# 01. Exploratory Data Analysis

## Introduction
This notebook is where I slow down and try to understand what is actually in the dataset before jumping into modeling.

The goal here is not to build anything yet. It is to get a feel for the market represented by the job listings, spot patterns that look useful, and notice the messy parts of the data that could hurt the model later.

So this notebook stays focused on:
- understanding the shape of the data
- looking at salary, experience, role, and location patterns
- calling out anomalies and weak signals
- ending with practical takeaways for the modeling notebook


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

DATA_PATH = Path('../data/job.csv')


## Data Overview
We start by loading the raw dataset and creating a lightly cleaned analytical copy. The preprocessing here is intentionally minimal: parse salary ranges, parse experience ranges, convert posting recency into days, and flag duplicates or remote roles.


In [ ]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.head()


In [ ]:
def parse_salary(text: str) -> pd.Series:
    text = str(text)
    if 'competitive salary' in text.lower():
        return pd.Series([pd.NA, pd.NA, False], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])

    nums = re.findall(r'[\d,]+', text)
    nums = [int(n.replace(',', '')) for n in nums]
    if not nums:
        return pd.Series([pd.NA, pd.NA, False], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])
    if len(nums) == 1:
        return pd.Series([nums[0], nums[0], True], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])
    return pd.Series([nums[0], nums[1], True], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])


def parse_posted_days(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower().replace('\n\n\nbe an early applicant', '')
    if 'today' in text or 'few hours' in text or 'just now' in text or 'immediately' in text:
        return 0

    match = re.search(r'(\d+)', text)
    number = int(match.group(1)) if match else None
    if 'week' in text:
        return (number or 1) * 7
    if 'month' in text:
        return (number or 1) * 30
    return number if number is not None else pd.NA


salary_parts = raw_df['ctc'].apply(parse_salary)
eda_df = pd.concat([raw_df.copy(), salary_parts], axis=1)
eda_df['avg_ctc'] = pd.to_numeric(eda_df['min_ctc'], errors='coerce').add(pd.to_numeric(eda_df['max_ctc'], errors='coerce')).div(2)
eda_df['min_exp'] = pd.to_numeric(eda_df['experience'].str.extract(r'(\d+)')[0], errors='coerce')
eda_df['max_exp'] = pd.to_numeric(eda_df['experience'].str.extract(r'(?:\d+)-(\d+)')[0], errors='coerce').fillna(eda_df['min_exp'])
eda_df['posted_days'] = eda_df['posted'].apply(parse_posted_days)
eda_df['is_remote'] = eda_df['location'].str.contains('work from home|remote', case=False, na=False)
eda_df['is_exact_duplicate'] = eda_df.duplicated(keep='first')
eda_df['is_noisy_duplicate'] = eda_df.duplicated(subset=['job_title', 'company_name', 'location'], keep='first')

analysis_df = eda_df.loc[~eda_df['is_noisy_duplicate']].copy()

summary = pd.DataFrame({
    'metric': [
        'Raw rows',
        'Rows after noisy duplicate removal',
        'Columns in raw data',
        'Salary disclosure rate',
        'Remote role rate',
        'Noisy duplicate rate',
    ],
    'value': [
        len(raw_df),
        len(analysis_df),
        raw_df.shape[1],
        round(float(analysis_df['is_salary_disclosed'].mean()), 3),
        round(float(analysis_df['is_remote'].mean()), 3),
        round(float(eda_df['is_noisy_duplicate'].mean()), 3),
    ],
})
summary


In [ ]:
overview = pd.DataFrame({
    'dtype': analysis_df.dtypes.astype(str),
    'missing_values': analysis_df.isna().sum(),
    'missing_pct': (analysis_df.isna().mean() * 100).round(2),
    'unique_values': analysis_df.nunique(dropna=True),
}).sort_values(['missing_pct', 'unique_values'], ascending=[False, False])
overview


### Data Overview: Interpretation
**Insight**: The dataset is large enough for analysis, but it contains duplicate-style postings and a non-trivial number of undisclosed salaries.

**Why it matters**: Raw row count alone overstates information content. Duplicate-like listings can bias frequency-based conclusions, while undisclosed salaries reduce the effective sample size for salary-focused analysis.

**Possible impact on modeling**: The next notebook should train on a deduplicated analytical dataset and treat missing salary values carefully rather than filling them blindly.


## Univariate Analysis
This section examines each major variable on its own to understand scale, skewness, coverage, and unusual values.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(analysis_df['avg_ctc'].dropna(), bins=40, ax=axes[0], color='#1f77b4')
axes[0].set_title('Average CTC Distribution')
axes[0].set_xlabel('Average CTC')

sns.boxplot(x=analysis_df['avg_ctc'].dropna(), ax=axes[1], color='#9ecae1')
axes[1].set_title('Average CTC Box Plot')
axes[1].set_xlabel('Average CTC')

plt.tight_layout()

analysis_df['avg_ctc'].describe()


### Salary Distribution: Interpretation
**Insight**: Salary is strongly right-skewed. The median salary is much lower than the maximum, and the presence of very small and very large values suggests outliers or inconsistent salary reporting.

**Why it matters**: A skewed target can distort averages and make the model overreact to rare high-salary listings.

**Possible impact on modeling**: The next notebook should examine target transformation or robust evaluation choices and inspect extreme salary values before final training.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

exp_order = analysis_df['experience'].value_counts().head(12)
sns.barplot(x=exp_order.values, y=exp_order.index, ax=axes[0], color='#4c78a8')
axes[0].set_title('Most Common Experience Labels')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Experience')

sns.histplot(analysis_df['posted_days'].dropna(), discrete=True, ax=axes[1], color='#f58518')
axes[1].set_title('Posting Recency Distribution')
axes[1].set_xlabel('Posted Days')

plt.tight_layout()

analysis_df[['min_exp', 'max_exp', 'posted_days']].describe()


### Experience and Posting Recency: Interpretation
**Insight**: Most listings cluster at lower experience levels, and posting recency is concentrated in the first three weeks. This looks like a fresher-heavy, recently scraped market snapshot rather than a balanced long-term market sample.

**Why it matters**: If the dataset is dominated by early-career roles and recent postings, any downstream model may underrepresent senior roles or long-tail hiring patterns.

**Possible impact on modeling**: The next notebook should keep an eye on class imbalance across experience levels and avoid over-interpreting posting age as a strong salary driver.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_jobs = analysis_df['job_title'].value_counts().head(10)
sns.barplot(x=top_jobs.values, y=top_jobs.index, ax=axes[0], color='#54a24b')
axes[0].set_title('Top 10 Job Titles')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Job Title')

top_locations = analysis_df['location'].value_counts().head(10)
sns.barplot(x=top_locations.values, y=top_locations.index, ax=axes[1], color='#e45756')
axes[1].set_title('Top 10 Locations')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Location')

plt.tight_layout()

pd.DataFrame({
    'top_jobs': top_jobs,
    'top_locations': top_locations,
})


### Role and Location Mix: Interpretation
**Insight**: The dataset is concentrated in a handful of commercial and creative roles, while a few cities and work-from-home dominate the location mix.

**Why it matters**: This is not a uniform job market sample. Strong concentration means global averages can hide important subgroup differences.

**Possible impact on modeling**: Job title and location are likely to be high-signal variables, but rare categories may need grouping to avoid unstable estimates.


## Bivariate Analysis
Now we look at relationships between variables to understand which patterns are likely to matter downstream and which ones may be noisy.


In [ ]:
corr_cols = ['min_exp', 'max_exp', 'posted_days', 'avg_ctc']
corr_matrix = analysis_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Matrix for Key Numeric Variables')
plt.tight_layout()

corr_matrix


### Correlation: Interpretation
**Insight**: Experience has a clearer positive relationship with salary than posting age does. In contrast, posted days has little to no direct linear relationship with salary.

**Why it matters**: Not every available numeric feature carries useful business signal. Weak relationships can still look important in a model if not validated carefully.

**Possible impact on modeling**: Experience should remain a core predictor, while posting age should be treated cautiously and validated against real generalization performance.


In [ ]:
exp_bucket = pd.cut(
    analysis_df['max_exp'],
    bins=[-1, 1, 3, 5, 10, 50],
    labels=['0-1', '2-3', '4-5', '6-10', '10+'],
)

exp_salary = analysis_df.groupby(exp_bucket, observed=False)['avg_ctc'].agg(['count', 'median', 'mean'])

plt.figure(figsize=(8, 5))
sns.barplot(x=exp_salary.index.astype(str), y=exp_salary['median'], color='#72b7b2')
plt.title('Median Salary by Experience Bucket')
plt.xlabel('Max Experience Bucket')
plt.ylabel('Median Salary')
plt.tight_layout()

exp_salary


### Salary vs Experience: Interpretation
**Insight**: Salary generally rises with experience, but the jump is not smooth across all buckets. Senior buckets have much higher pay but far fewer observations.

**Why it matters**: Higher salary at higher experience levels is directionally expected, but sparse senior data makes those estimates noisier.

**Possible impact on modeling**: The next notebook should preserve experience as a key feature, but also avoid treating extreme senior segments as equally reliable as the bulk of the data.


In [ ]:
location_salary = (
    analysis_df.groupby('location')
    .agg(posts=('location', 'size'), median_salary=('avg_ctc', 'median'), disclosed_rate=('is_salary_disclosed', 'mean'))
    .sort_values('posts', ascending=False)
    .head(10)
)

remote_salary = analysis_df.groupby('is_remote')['avg_ctc'].agg(['count', 'median', 'mean'])
job_salary = (
    analysis_df.groupby('job_title')
    .agg(posts=('job_title', 'size'), median_salary=('avg_ctc', 'median'))
    .query('posts >= 5')
    .sort_values('median_salary', ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=location_salary['median_salary'], y=location_salary.index, ax=axes[0], color='#b279a2')
axes[0].set_title('Median Salary for Top 10 Locations by Volume')
axes[0].set_xlabel('Median Salary')
axes[0].set_ylabel('Location')

sns.barplot(x=job_salary['median_salary'], y=job_salary.index, ax=axes[1], color='#ff9da6')
axes[1].set_title('Highest Median Salary Roles (min 5 posts)')
axes[1].set_xlabel('Median Salary')
axes[1].set_ylabel('Job Title')

plt.tight_layout()

location_salary, remote_salary, job_salary


### Salary vs Role and Location: Interpretation
**Insight**: Salary varies materially by both role and location. Remote roles are not obviously discounted in this sample, and some technical roles sit well above the dataset median.

**Why it matters**: Salary is clearly market-specific rather than universal. Geography and role identity capture structural pay differences.

**Possible impact on modeling**: Role and location should remain central predictors, but rare high-paying roles may need grouping or regularization to reduce volatility.


In [ ]:
salary_anomalies = analysis_df.loc[
    (analysis_df['avg_ctc'].notna()) & ((analysis_df['avg_ctc'] < 50000) | (analysis_df['avg_ctc'] > 1500000)),
    ['job_title', 'company_name', 'location', 'ctc', 'avg_ctc', 'experience', 'posted']
].sort_values('avg_ctc')

posted_salary = analysis_df.groupby(
    pd.cut(analysis_df['posted_days'], bins=[-1, 1, 7, 14, 30, 365], labels=['0-1', '2-7', '8-14', '15-30', '30+']),
    observed=False,
)['avg_ctc'].agg(['count', 'median', 'mean'])

salary_anomalies.head(15), posted_salary


### Anomalies and Edge Cases: Interpretation
**Insight**: The data contains extreme salary values at both ends, and posting recency does not show a stable salary trend. That combination suggests some values reflect noise, formatting issues, or narrow submarkets rather than a broad pattern.

**Why it matters**: Unchecked anomalies can dominate error metrics, while weak but noisy variables can create brittle model behavior.

**Possible impact on modeling**: The next notebook should inspect salary outliers explicitly and validate whether recency deserves inclusion at all.


## Key Insights
1. Salary is heavily right-skewed, so averages alone are not representative of the market.
2. Experience has a stronger relationship with salary than posting age.
3. Job title and location are likely to carry major predictive signal because salary varies across both.
4. Duplicate-style listings and undisclosed salaries reduce the usable analytical sample.
5. The dataset is concentrated in early-career roles and recent postings, so it may not generalize evenly across all hiring segments.
6. Extreme salary values and weak recency signal should be treated as modeling risks rather than trusted patterns.


## EDA Summary
### What I want to carry forward into the next notebook
- Use the deduplicated analytical dataset instead of the raw scrape.
- Be careful with salary outliers, because they can pull the model around more than they should.
- Keep experience, role, and location as the core predictors.
- Treat posting recency with caution, because it looks noisy and inconsistent.
- Remember that rare roles and senior segments have less support, so their patterns may be less stable.

In short: the data is usable, but it rewards simple, careful choices more than clever ones.
